# Shot-by-Shot Video Analysis

Local LLM + Faster-Whisper pipeline for video analysis on Kaggle.

## Requirements
- GPU: T4 x2 (enable in Runtime → Change runtime type → GPU T4 x2)
- Internet: Enable for first-time model download

## Workflow
1. **T4x2 OFF**: Run Cell 1 (install) → Cells 2-3 (imports + config) → Cell A (download + upload to Kaggle Dataset)
2. **Turn T4x2 ON** — kernel restarts automatically
3. **Change MODEL_ID** in CONFIGURATION to `/kaggle/input/yoofun/qwen3-vl-30b/`
4. Run from Cell 2 onward (weights load from Kaggle Dataset at disk speed)

## Configuration
Edit Cell 3 to set your preferences.

## Output Files
- `shot_by_shot_output.csv` - Main merged output
- `stage1_descriptions.csv` - Detailed VLM descriptions
- `stage2_audio_descriptions.csv` - Concise AD sentences

In [ ]:
# Install dependencies
# Run this cell ONCE, then manually restart: Run → Restart session
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"

# 1. Force the correct NumPy version first
!pip install -q --no-cache-dir "numpy==1.26.4" --force-reinstall --no-deps

# 2. Install core utilities
!pip install -q --no-cache-dir scenedetect "opencv-python-headless>=4.10"

# 3. Cleanly install Unsloth and all matching backend dependencies together
!pip install --no-cache-dir "transformers>=4.51.0" "accelerate>=1.0" "huggingface_hub>=1.22" "bitsandbytes>=0.46.1"
!pip install --no-cache-dir unsloth unsloth_zoo

# 4. Check versions using Linux 'grep' instead of Windows 'findstr'
print("\n--- Verifying Installed Versions ---")
!pip show bitsandbytes | grep Version
!pip show scenedetect | grep Version
!pip show unsloth | grep Version

print("\nDONE. Now do: Run → Restart session, then run from Cell 2")


In [1]:
# Import libraries
import os
# Retrieve HF_TOKEN from Kaggle Secrets for faster authenticated downloads
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

import json
import torch
import pandas as pd
import numpy as np
import cv2
import base64
from PIL import Image
from scenedetect import detect, AdaptiveDetector

# Check GPU
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.device_count()} device(s)")
    for i in range(torch.cuda.device_count()):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("WARNING: No GPU detected. Enable GPU T4 x2 in Runtime settings.")
    
    !pip install -U "bitsandbytes>=0.46.1"                                                                         


GPU available: 2 device(s)
  Device 0: Tesla T4
  Device 1: Tesla T4


In [8]:
# CONFIGURATION - Edit these settings
# ============================================

# --- MODEL PATH ---
# After uploading to Kaggle Dataset, use the local path below.
# First time: the dataset gets mounted at /kaggle/input/qwen3-vl-30b/
MODEL_ID = "unsloth/Qwen3-VL-32B-Instruct-unsloth-bnb-4bit"

# Video path (upload video to Kaggle Datasets first)
VIDEO_PATH = "/kaggle/input/datasets/yoofun/whitesummer"

# Whisper settings
USE_WHISPER = False              # Set False to skip transcription
WHISPER_MODEL = "medium"        # tiny, base, small, medium, large-v3
WHISPER_LANGUAGE = None          # None = auto-detect, or "en", "zh", etc.

# LLM settings (local Qwen3-VL-30B on dual T4)
USE_VLM = True                  # Set False to skip VLM descriptions
USE_STAGE2 = True               # Set False to skip Stage 2 summarization
VIDEO_TYPE = "movie"            # "movie" or "tv_series"

# ============================================

In [25]:
# Download model & save as permanent Kaggle Dataset (run ONCE, T4x2 OFF)
import os
import shutil
import json
import subprocess

# 1. 徹底清空先前所有失敗嘗試留下的髒快取，避免佔用原本就緊張的 20GB 磁碟空間
print("Cleaning old cache files...")
for path in ["/kaggle/working/qwen_model", "/kaggle/tmp/qwen_model",
             "/kaggle/tmp/hf_cache", os.path.expanduser("~/.cache/huggingface")]:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)

# 2. 定義模型與暫存路徑
MODEL = "unsloth/Qwen3-VL-8B-Thinking-unsloth-bnb-4bit"
DST = "/kaggle/tmp/qwen_model"
os.makedirs(DST, exist_ok=True)

# 3. 確保 huggingface_hub 是最新版且支援最新的 'hf' 命令列
!pip install -q --upgrade huggingface_hub

# 4. 呼叫現代化的 'hf download' 架構進行高效下載
print(f"Downloading {MODEL} to {DST}...")
# 使用全新的 hf download 語法取代過時的 huggingface-cli
ret = subprocess.call(["hf", "download", MODEL, "--local-dir", DST])
if ret != 0:
    raise RuntimeError(f"Download failed (exit {ret})")

# 5. 驗證檔案數量
files = os.listdir(DST)
print(f"Success! Downloaded {len(files)} files into {DST}")

# 6. 自動建立正確對齊 8B Thinking 命名規格的 Kaggle Metadata 設定檔
print("Creating Kaggle Dataset metadata config...")
# 注意：這裡的 id 'yoofun/qwen3-vl-8b-thinking' 必須與你在 Kaggle 上預先建立的名字、
# 或者你想發布的名字完全一致。這裡已幫你將原本的 32B 改為正確的 8b-thinking
metadata = {
    "title": "qwen3-vl-8b-thinking",
    "id": "yoofun/qwen3-vl-8b-thinking",
    "licenses": [{"name": "Apache-2.0"}]
}

with open(os.path.join(DST, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

# 7. 呼叫 Kaggle API 自動壓縮並打包上傳至 Kaggle Dataset 伺服器
# 注意：若這是你第一次建立此 Dataset，請確保它是一個全新的 id 名稱
print("Pushing to Kaggle Datasets server...")
!kaggle datasets create -p {DST} --dir-mode zip

print("\nALL DONE! You can now permanently add this dataset from your Data sidebar panel.")


Cleaning old cache files...


Fetching 16 files: 100%|██████████| 16/16 [00:29<00:00,  1.86s/it]


✓ Downloaded
  path: /kaggle/tmp/qwen_model
Success! Downloaded 17 files into /kaggle/tmp/qwen_model
Creating Kaggle Dataset metadata config...
Pushing to Kaggle Datasets server...
Starting upload for file merges.txt
100%|██████████████████████████████████████| 1.59M/1.59M [00:00<00:00, 3.93MB/s]
Upload successful: merges.txt (2MB)
Starting upload for file generation_config.json
100%|████████████████████████████████████████████| 192/192 [00:00<00:00, 497B/s]
Upload successful: generation_config.json (192B)
Starting upload for file special_tokens_map.json
100%|██████████████████████████████████████████| 614/614 [00:00<00:00, 1.71kB/s]
Upload successful: special_tokens_map.json (614B)
Starting upload for file config.json
100%|██████████████████████████████████████| 4.45k/4.45k [00:00<00:00, 11.5kB/s]
Upload successful: config.json (4KB)
Starting upload for file tokenizer.json
100%|██████████████████████████████████████| 10.9M/10.9M [00:00<00:00, 22.1MB/s]
Upload successful: tokenizer.jso

In [ ]:
# Load Faster-Whisper model
whisper_model = None

if USE_WHISPER:
    from faster_whisper import WhisperModel
    
    print(f"Loading Whisper model: {WHISPER_MODEL}")
    # Use float16 on GPU, int8 on CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    whisper_model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)
    print(f"Whisper loaded on {device} with {compute_type}")
else:
    print("Whisper disabled")

In [17]:
# Load LLM using Unsloth (Optimized for bnb-4bit 8B Model)
import os
import torch
import bitsandbytes as bnb

# 1. Enforce environment paths to avoid Kaggle's 20GB disk limit
os.environ["HF_HOME"] = "/tmp/hf_cache"

print(f"bitsandbytes version: {bnb.__version__}")

# Update this to the Unsloth 4-bit 7B/8B model variant
# Note: For Qwen VL, the primary optimized variant is the 7B/7.2B architecture
MODEL_ID = "unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit"

llm_model = None
llm_processor = None

if USE_VLM or USE_STAGE2:
    # 2. Import the correct Unsloth vision engine
    from unsloth import FastVisionModel
    
    print(f"Loading LLM (bnb-4bit via Unsloth): {MODEL_ID}")
    print(f"GPU count: {torch.cuda.device_count()}")
    
    # 3. Load model smoothly into a single GPU instance
    llm_model, llm_processor = FastVisionModel.from_pretrained(
        model_name = MODEL_ID,
        load_in_4bit = True,         # Enforces the 4-bit quantization layout
        low_cpu_mem_usage = True,    # Prevents Kaggle 16GB CPU RAM crash
    )
    
    # 4. Activate inference optimization to save more VRAM during generation
    llm_model = FastVisionModel.for_inference(llm_model)
    
    print("LLM loaded successfully via Unsloth!")
else:
    print("VLM/Stage2 disabled, skipping LLM load")


bitsandbytes version: 0.49.2
Loading LLM (bnb-4bit via Unsloth): unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit
GPU count: 2
==((====))==  Unsloth 2026.6.9: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

LLM loaded successfully via Unsloth!


In [11]:
# Shot detection
import os
import pandas as pd
from scenedetect import detect, AdaptiveDetector

# 1. Inspect the directory and dynamically locate the actual video file
TARGET_DIR = "/kaggle/input/datasets/yoofun/whitesummer"
print(f"Scanning target directory: {TARGET_DIR}")

if not os.path.exists(TARGET_DIR):
    raise FileNotFoundError(f"CRITICAL ERROR: The directory '{TARGET_DIR}' does not exist!")

# Find any file ending with common video extensions
video_extensions = (".mp4", ".mkv", ".avi", ".mov", ".flv", ".webm")
found_videos = [f for f in os.listdir(TARGET_DIR) if f.lower().endswith(video_extensions)]

if not found_videos:
    # Print what is actually inside the folder to help you debug
    print(f"Contents of directory: {os.listdir(TARGET_DIR)}")
    raise FileNotFoundError("CRITICAL ERROR: No valid video files found inside the specified directory!")

# Automatically pick the first video file found inside the folder
REAL_VIDEO_PATH = os.path.join(TARGET_DIR, found_videos[0])
file_size_mb = os.path.getsize(REAL_VIDEO_PATH) / (1024 * 1024)
print(f"Success! Found Video: '{found_videos[0]}' ({file_size_mb:.2f} MB)")


# 2. Configured shot detection function
def detect_shots(video_path, threshold=27.0):
    try:
        print("Initializing shot detector using primary backend...")
        scene_list = detect(video_path, AdaptiveDetector(adaptive_threshold=threshold))
    except Exception as cv_error:
        print(f"\nPrimary backend failed: {cv_error}")
        print("Attempting automatic failover to PyAV decoder framework...")
        
        import subprocess
        subprocess.run(["pip", "install", "-q", "av"], check=True)
        
        scene_list = detect(
            video_path, 
            AdaptiveDetector(adaptive_threshold=threshold), 
            backend="pyav"
        )

    shots = []
    for idx, (start, end) in enumerate(scene_list):
        shots.append({
            "shot_id": idx + 1,
            "start_time": start.get_seconds(),
            "end_time": end.get_seconds()
        })
    return shots


# 3. Execute processing chain using the resolved file path
shots = detect_shots(REAL_VIDEO_PATH)
print(f"\nExecution Complete: Found {len(shots)} shots total.")

# 4. Display the results
df_shots = pd.DataFrame(shots)
df_shots


[pyscenedetect|INFO]Detecting scenes...


Scanning target directory: /kaggle/input/datasets/yoofun/whitesummer
Success! Found Video: 'White Summer Live  1080.mp4' (50.48 MB)
Initializing shot detector using primary backend...


[py.warnings|WARNING]/tmp/ipykernel_1551/242050568.py:50: DeprecationWarning: get_seconds() is deprecated, use the `seconds` property instead.
  "start_time": start.get_seconds(),

[py.warnings|WARNING]/tmp/ipykernel_1551/242050568.py:51: DeprecationWarning: get_seconds() is deprecated, use the `seconds` property instead.
  "end_time": end.get_seconds()




Execution Complete: Found 6 shots total.


,shot_id,start_time,end_time
0,1,0.000000,65.398667
1,2,65.398667,95.895800
2,3,95.895800,149.782967
3,4,149.782967,157.757600
4,5,157.757600,184.884700
5,6,184.884700,235.201633


In [ ]:
# Whisper transcription
def transcribe_video(video_path, model, language=None):
    segments, info = model.transcribe(
        video_path,
        language=language,
        beam_size=5,
        vad_filter=True
    )
    
    subtitles = []
    for seg in segments:
        subtitles.append({
            "text": seg.text.strip(),
            "start_time": seg.start,
            "end_time": seg.end
        })
    return subtitles

def find_dialogue_gaps(subtitles, shots, min_gap_duration=0.5):
    """Find time intervals without dialogue for AD placement."""
    gaps = []
    for shot in shots:
        shot_start = shot["start_time"]
        shot_end = shot["end_time"]
        
        shot_subs = [s for s in subtitles 
                     if s["start_time"] < shot_end and s["end_time"] > shot_start]
        shot_subs.sort(key=lambda x: x["start_time"])
        
        current_time = shot_start
        for sub in shot_subs:
            if sub["start_time"] > current_time + min_gap_duration:
                gaps.append({
                    "shot_id": shot["shot_id"],
                    "start_time": current_time,
                    "end_time": sub["start_time"]
                })
            current_time = max(current_time, sub["end_time"])
        
        if shot_end > current_time + min_gap_duration:
            gaps.append({
                "shot_id": shot["shot_id"],
                "start_time": current_time,
                "end_time": shot_end
            })
    return gaps

if USE_WHISPER and whisper_model:
    print("Transcribing video...")
    subtitles = transcribe_video(VIDEO_PATH, whisper_model, WHISPER_LANGUAGE)
    print(f"Found {len(subtitles)} subtitle segments")
    
    dialogue_gaps = find_dialogue_gaps(subtitles, shots)
    print(f"Found {len(dialogue_gaps)} dialogue gaps for AD placement")
else:
    print("Whisper disabled")
    subtitles = []
    dialogue_gaps = []

In [ ]:
# Merge shots and subtitles
def merge_shots_subtitles(shots, subtitles):
    results = []
    for shot in shots:
        overlapping = [s for s in subtitles 
                      if s["start_time"] < shot["end_time"] and s["end_time"] > shot["start_time"]]
        subtitle_text = " ".join([s["text"] for s in overlapping]) if overlapping else ""
        results.append({
            "shot_id": shot["shot_id"],
            "start_time": shot["start_time"],
            "end_time": shot["end_time"],
            "subtitle": subtitle_text
        })
    return pd.DataFrame(results)

merged_df = merge_shots_subtitles(shots, subtitles)
merged_df

In [18]:
    # VLM description (Stage 1) using transformers
import pandas as pd

# 建立基礎的分鏡 DataFrame，供後續 VLM 階段填入資料
if 'shots' in locals() and shots:
        merged_df = pd.DataFrame(shots)
        print("成功建立 merged_df！目前的結構如下：")
        print(merged_df.head())
else:
        # 萬一先前的 shots 沒有成功執行，這裡提供一組模擬資料預防崩潰
        print("警告：未偵測到變數 'shots'，請確保前一個分鏡偵測儲存格有成功執行。")
        # 建立一個空的或預設結構
        merged_df = pd.DataFrame(columns=["shot_id", "start_time", "end_time"])

import numpy as np
    import cv2
    import torch
    import warnings
    from PIL import Image
    
    # Suppress the text generation priority warning to keep your console clean
    warnings.filterwarnings("ignore", message=".*Both max_new_tokens and max_length seem to have been set.*")
    
    # VLM description (Stage 1) using transformers
    def extract_frames(video_path, shot, num_frames=8):
        """Extract frames from video shot as PIL Images."""
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        start_frame = int(shot["start_time"] * fps)
        end_frame = int(shot["end_time"] * fps)
        frame_indices = np.linspace(start_frame, end_frame, num_frames, dtype=int)
        
        frames = []
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(frame_rgb))
        cap.release()
        return frames
    
    def describe_frames_local(frames, model, processor):
        """Get VLM description using Qwen3-VL/Qwen2.5-VL via Unsloth."""
        prompt = "請簡短描述這段影片片段發生了什麼事。請使用繁體中文。專注於角色、動作和環境。"
        
        content = [{"type": "text", "text": prompt}]
        for frame in frames:
            content.append({"type": "image", "image": frame})
        messages = [{"role": "user", "content": content}]
        
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=256)
        
        generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, output_ids)]
        response = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        return response
    
    def summarize_to_ad_local(desc, model, processor, word_limit=15):
        """Summarize description to concise AD sentence using Qwen3-VL/Qwen2.5-VL."""
        prompt = f"""請將以下描述濃縮成一句簡潔的口述影像句子。使用繁體中文。
    專注於角色、動作和關鍵物件。
    使用名字或代名詞。避免提到鏡頭。
    限制在 {word_limit} 個字以內。
    
    Input: {desc}
    
    Output:"""
        
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=128)
        
        generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, output_ids)]
        result = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0].strip()
        if not result.endswith("."):
            result += "."
        return result
    
    # Run VLM if enabled
    descriptions_dict = {}
    if USE_VLM and llm_model:
        print("Running Stage 1: VLM descriptions...")
        for shot in shots:
            try:
                # FIX: We use REAL_VIDEO_PATH which we found from the last cell instead of the directory string
                frames = extract_frames(REAL_VIDEO_PATH, shot) 
                desc = describe_frames_local(frames, llm_model, llm_processor)
                descriptions_dict[shot["shot_id"]] = desc
                print(f"Shot {shot['shot_id']}: OK")
            except Exception as e:
                print(f"Shot {shot['shot_id']}: Failed - {e}")
                descriptions_dict[shot["shot_id"]] = ""
        
        merged_df["video_description"] = merged_df["shot_id"].map(descriptions_dict).fillna("")
        
        stage1_df = merged_df.copy()
        stage1_df.to_csv("/kaggle/working/stage1_descriptions.csv", index=False)
        print("Stage 1 saved to: stage1_descriptions.csv")
    else:
        print("VLM disabled")
    
    # Run Stage 2 if enabled
    if USE_STAGE2 and llm_model and descriptions_dict:
        print("\nRunning Stage 2: Summarizing to AD...")
        ad_sentences = []
        for _, row in merged_df.iterrows():
            duration = row["end_time"] - row["start_time"]
            word_limit = max(1, int(duration * 3))  # 3 words per second
            try:
                ad = summarize_to_ad_local(row["video_description"], llm_model, llm_processor, word_limit)
                ad_sentences.append(ad)
            except:
                ad_sentences.append("")
        
        merged_df["ad_sentence"] = ad_sentences
        
        stage2_df = merged_df[["shot_id", "start_time", "end_time", "ad_sentence"]].copy()
        stage2_df.to_csv("/kaggle/working/stage2_audio_descriptions.csv", index=False)
        print("Stage 2 saved to: stage2_audio_descriptions.csv")
    else:
        print("Stage 2 disabled")
    
    merged_df


成功建立 merged_df！目前的結構如下：
   shot_id  start_time    end_time
0        1    0.000000   65.398667
1        2   65.398667   95.895800
2        3   95.895800  149.782967
3        4  149.782967  157.757600
4        5  157.757600  184.884700
Running Stage 1: VLM descriptions...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Shot 1: OK


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Shot 2: OK


...

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Shot 6: OK
Stage 1 saved to: stage1_descriptions.csv

Running Stage 2: Summarizing to AD...


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Stage 2 saved to: stage2_audio_descriptions.csv


,shot_id,start_time,end_time,video_description,ad_sentence
0,1,0.000000,65.398667,舞台上，一位穿着白色太空服的表演者从一个圆形开口处走出，周围有几位穿着类似服装的舞者在跳舞。...,舞台上，表演者穿白太空服，從圓形開口處走出，周圍舞者跳舞，觀眾揮動螢光棒，營造星光璀璨效果。.
1,2,65.398667,95.895800,在這個場景中，一位穿著白色禮服的歌手站在一個現代感十足的舞台上，背後有一個圓形的開口，透過開...,歌手專注於表演，周圍有太空服舞者和閃爍燈光，營造未來科技氛圍。.
2,3,95.895800,149.782967,這段影片片段中，一位穿著白色禮服的歌手在舞台上表演。他手持麥克風，表情投入地唱歌。舞台背景燈...,一位穿著白色禮服的歌手在舞台上表演，手持麥克風，表情投入地唱歌，舞台背景燈光變化多端，觀眾席...
3,4,149.782967,157.757600,在這個片段中，一位穿著白色禮服的歌手站在舞台上，手持麥克風正在演唱。他戴著眼鏡，表情投入，時...,歌手投入演唱，舞台燈光變化夢幻。.
4,5,157.757600,184.884700,在這個場景中，一位穿著白色禮服的歌手站在舞台上，手持麥克風，正在演唱。舞台背景是深色的，有藍...,歌手在舞台上演唱，舞者輕盈起舞，背景大屏幕顯示特寫畫面。.
5,6,184.884700,235.201633,在這段影片片段中，一位穿著白色禮服的歌手正在舞台上表演。他手持麥克風，表情投入地唱歌。舞台燈...,一位穿著白色禮服的歌手在舞台上表演，手持麥克風，表情投入地唱歌，舞台燈光閃爍，背景中有一個巨...


In [22]:
# VLM description (Gemini)
import os
import time
import pandas as pd

# 1. 初始化 merged_df 資料表結構
if 'shots' in locals() and shots:
    merged_df = pd.DataFrame(shots)
    print("成功初始化 merged_df 結構！")
else:
    print("警告：未偵測到變數 'shots'，將使用預設空欄位初始化。")
    merged_df = pd.DataFrame(columns=["shot_id", "start_time", "end_time"])

# 2. 自動檢查並安裝 Google GenAI SDK
try:
    from google import genai
    from google.genai import types
except ImportError:
    print("正在安裝 Google GenAI 官方 SDK...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "google-genai"], check=True)
    from google import genai
    from google.genai import types

# 3. 初始化 Gemini 客戶端
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    # 萬一 Kaggle Secrets 沒設定，請把你的 API Key 貼在下方引號內
    api_key = "YOUR_GEMINI_API_KEY_HERE"

client = genai.Client(api_key=api_key)

# 綁定 Google AI Studio 免費版支援最新的 3.5 核心
TARGET_MODEL = "gemini-3.5-flash"
descriptions_dict = {}

# --- STAGE 1: 使用 Gemini 3.5 Flash 進行全影片分鏡分析 ---
if USE_VLM:
    print(f"正在將影片上傳至 Gemini AI Studio 伺服器: {REAL_VIDEO_PATH}")
    video_upload = client.files.upload(file=REAL_VIDEO_PATH)
    
    # 等待雲端影片解碼與分析完成
    print("等待 Gemini 進行影片串流解析中...")
    while video_upload.state.name == "PROCESSING":
        time.sleep(4)
        video_upload = client.files.get(name=video_upload.name)
        
    if video_upload.state.name == "FAILED":
        raise ValueError(f"Gemini 影片解析失敗: {video_upload.error.message}")
    print("影片解析完成！開始逐個分鏡進行分析...")

    print("\n[開始執行 Stage 1] 正在生成分鏡描述描述...")
    for shot in shots:
        try:
            # 建立明確的時間範圍 Prompt 指令
            prompt = f"""
            你是一個專業的影視分析與口述影像專家。
            請專注看影片中「第 {shot['start_time']:.2f} 秒」到「第 {shot['end_time']:.2f} 秒」之間的片段。
            請精準、簡短描述這段期間畫面裡發生了什麼事。請使用繁體中文。
            請專注於：有哪些角色、他們的具體肢體動作、以及環境/物件的變化。
            絕對不要提及這個時間區間之外的任何畫面。
            """
            
            # 免費版核心：直接傳入 video_upload 物件指標與 Prompt，不使用快取
            response = client.models.generate_content(
                model=TARGET_MODEL,
                contents=[video_upload, prompt]
            )
            
            descriptions_dict[shot["shot_id"]] = response.text.strip()
            print(f"Shot {shot['shot_id']}: OK")
            
            # 🔥 關鍵防爆機制：每次呼叫後強制暫停 5 秒，確保完美的 12 RPM Pacing，絕對不踩 15 RPM 限制
            time.sleep(5)
            
        except Exception as e:
            print(f"Shot {shot['shot_id']}: 失敗 - {e}")
            descriptions_dict[shot["shot_id"]] = ""
            # 如果失敗，也稍微休息一下防止連續報錯
            time.sleep(5)
            
    # 任務完成後，安全清理雲端暫存的影片檔案
    client.files.delete(name=video_upload.name)
    print("\n雲端暫存影片檔案已安全清理。")

    # 將描述寫回 Pandas DataFrame 並存成 CSV
    merged_df["video_description"] = merged_df["shot_id"].map(descriptions_dict).fillna("")
    merged_df.to_csv("/kaggle/working/stage1_descriptions.csv", index=False)
    print("Stage 1 成果已儲存至: stage1_descriptions.csv")
else:
    print("VLM 描述功能已停用。")


# --- STAGE 2: 口述影像（Audio Description）精簡濃縮 ---
if USE_STAGE2 and (descriptions_dict or "video_description" in merged_df.columns):
    print("\n[開始執行 Stage 2] 正在濃縮成口述影像句子...")
    ad_sentences = []
    
    for idx, row in merged_df.iterrows():
        desc_payload = row["video_description"] if "video_description" in merged_df.columns else descriptions_dict.get(row["shot_id"], "")
        
        if not desc_payload:
            ad_sentences.append("")
            continue
            
        # 根據分鏡秒數動態限制字數 (每秒約 3 個字)
        duration = row["end_time"] - row["start_time"]
        word_limit = max(1, int(duration * 3))  
        
        prompt = f"""
        請將以下描述濃縮成一句最精煉、流暢的口述影像句子（給視障者聽的畫面旁白）。使用繁體中文。
        專注於角色、核心動作和關鍵物件。
        請直接描述畫面，使用名字或代名詞。避免提到「鏡頭」、「畫面」、「切換」、「影片」等詞彙。
        請嚴格限制在 {word_limit} 個字以內。

        Input: {desc_payload}

        Output:"""
        
        try:
            # 這裡同樣使用超快的 3.5 Flash 進行文字純文字濃縮
            response = client.models.generate_content(
                model=TARGET_MODEL,
                contents=[prompt]
            )
            result = response.text.strip()
            if not result.endswith("."):
                result += "."
            ad_sentences.append(result)
            
            # 純文字階段同樣加入 3 秒暫停防爆限制
            time.sleep(3)
        except Exception as e:
            print(f"Shot {row['shot_id']} 濃縮失敗: {e}")
            ad_sentences.append("")
            time.sleep(3)
            
    merged_df["ad_sentence"] = ad_sentences
    stage2_df = merged_df[["shot_id", "start_time", "end_time", "ad_sentence"]].copy()
    stage2_df.to_csv("/kaggle/working/stage2_audio_descriptions.csv", index=False)
    print("Stage 2 成果已儲存至: stage2_audio_descriptions.csv")
else:
    print("Stage 2 濃縮功能已停用。")

# 最終在 Notebook 畫面上印出整合後的完整報表
merged_df


成功初始化 merged_df 結構！
正在將影片上傳至 Gemini AI Studio 伺服器: /kaggle/input/datasets/yoofun/whitesummer/White Summer Live  1080.mp4
等待 Gemini 進行影片串流解析中...
影片解析完成！開始逐個分鏡進行分析...

[開始執行 Stage 1] 正在生成分鏡描述描述...
Shot 1: OK
Shot 2: OK
Shot 3: OK
Shot 4: OK
Shot 5: OK
Shot 6: OK

雲端暫存影片檔案已安全清理。
Stage 1 成果已儲存至: stage1_descriptions.csv

[開始執行 Stage 2] 正在濃縮成口述影像句子...
Stage 2 成果已儲存至: stage2_audio_descriptions.csv


,shot_id,start_time,end_time,video_description,ad_sentence
0,1,0.000000,65.398667,在這段影片（0:00 - 1:05.40）中，呈現了演唱會的開場演出：\n\n* **環...,在滿是紫色螢光棒海的體育館中，舞台中央巨大的白色正方體緩緩旋轉，多位身穿白衣、戴著圓形凹面面...
1,2,65.398667,95.895800,在這段片段中，畫面呈現了以下的舞台表演過程：\n\n* **歌手演唱**：一名染金髮、戴眼鏡...,身著白西裝的金髮男歌手在舞台中央巨大白色立方體的圓孔中持白色麥克風高歌，周圍戴著巨大白色球形...
2,3,95.895800,149.782967,在這段期間，身穿裝飾有金屬配飾的白色西裝、戴著眼鏡的男歌手，在舞台中央手持麥克風深情歌唱。\...,身穿白西裝的男歌手在升降平台上深情歌唱，舞台中央的白色立方體隨之展開傾斜，數名戴著白色球形頭...
3,4,149.782967,157.757600,在這段期間，身穿點綴亮片白色西裝、戴著眼鏡的金髮男歌手站在舞台上。他右手握著白色麥克風深情演...,身穿白西裝的金髮歌手，手握麥克風深情演唱。.
4,5,157.757600,184.884700,在這段期間，畫面呈現以下內容：\n\n* **角色與肢體動作**：身穿白色西裝的男歌手在舞台...,身穿白西裝的男歌手在圓形舞台中央唱歌，周圍舞者戴著白圓罩律動；上方旋轉的圓柱螢幕與地面的圓形...
5,6,184.884700,235.201633,在這段期間，畫面中發生了以下事件：\n\n* **184.88秒 - 194.00秒**：戴...,身穿白色西裝的男歌手在移動的幾何平台上深情獻唱，伴隨戴著白面罩的舞群起舞、漫天飄落的紫色紙花...


In [23]:
# Save outputs
output_path = "/kaggle/working/shot_by_shot_output.csv"
merged_df.to_csv(output_path, index=False)
print(f"Main output saved to: {output_path}")

print("\nOutput files:")
print("  - shot_by_shot_output.csv")
if USE_VLM and llm_model:
    print("  - stage1_descriptions.csv")
if USE_STAGE2 and llm_model:
    print("  - stage2_audio_descriptions.csv")

Main output saved to: /kaggle/working/shot_by_shot_output.csv

Output files:
  - shot_by_shot_output.csv
  - stage1_descriptions.csv
  - stage2_audio_descriptions.csv


In [24]:
# Download links
from IPython.display import FileLink, display

print("Download links:")
display(FileLink("/kaggle/working/shot_by_shot_output.csv"))
if USE_VLM and llm_model:
    display(FileLink("/kaggle/working/stage1_descriptions.csv"))
if USE_STAGE2 and llm_model:
    display(FileLink("/kaggle/working/stage2_audio_descriptions.csv"))

Download links:


/kaggle/working/shot_by_shot_output.csv

/kaggle/working/stage1_descriptions.csv

/kaggle/working/stage2_audio_descriptions.csv